# Eval Overview - Gene Expression Prediction

we run two types of evals for gene expression: sample level and perturbation level. 
`<TO UPDATE>`

Sample-level (top-20): computed per individual cell during the inference loop (line 534). Each
  cell has its own expression profile, so the top-20 DEGs are specific to that cell's actual
  response. This gives you a noisy but granular view.

Perturbation-level (top-50): computed on the mean profile across all replicates of that
  perturbation (line 1228). Averaging first washes out cell-to-cell noise and reveals the
  perturbation's true effect. The top-50 genes selected from the mean delta are the genes that
  consistently respond to this perturbation, not genes that happened to spike in one cell due to
  technical noise.



In [1]:
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import r2_score
import torch

In [2]:
SEED = 1337
np.random.seed(SEED)

## Data Prep
We'll start by preparing our data. For this evaluation, we need to have the true cell expressions, the control expression for each cell, and the perturbation information. To show both sample level and perturbation level we'll mock up 6 samples across 4 perturbations.  A "perturbation" is a unique combination of a sequence, target, modality, and mode applied to a cell type. A perturbation can span datasets.  For the 6 samples we'll show the real control cell expression, predicted control cell express (running the student encoder and then linear decoder),  real case cell expression, predicted case cell expression, and the perturbation.  We'll go ahead and do all 6 samples in batches.  


For predicted expression, we use the data created by the [linear expression decoder](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_decoders.ipynb). As a reminder, the decoder takes the mean (mu) BioJEPA-AC output based on the perturbations and cell expression pattern, and then uses a linear layer to project down to a $[\text{n\_genes},\text{1}]$ matrix with a single value per gene representing the expression. 

We'll stage data to show a few different predictions: a strong prediction, weak prediction, inverse prediction, and over prediction.  

In [3]:
unique_perts = 4
num_genes = 8
num_cells = 6
TOP_K = 3

**Pertubations**   

We'll first start with our perturbations. Even though we have 6 cells, our sample will only have 4 unique perturbations. To define a unique perturbations, it's not just about what we target, but the context of it.  Because of this, we represent a unique perturbations as $\text{(seq id, targ id, modality id, mode id, cell type)}$.  IDs are used since our model keeps the perturbation information in separate caches from our sample expression counts to avoid heavy duplication of infromation.

We'll also create a mapping of the sample to the perturbation, where the first two samples map to the first perturbation, the next two samples to the second perturbation, and then the final two samples  each have a unique perturbation. 

In [4]:
pert_keys = [                                                                      
      (0, 0, 0, 0, 0),  # pert 0: DNA CRISPRi, cell type 0 
      (1, 1, 0, 0, 0),  # pert 1: DNA CRISPRi, cell type 0 
      (2, 2, 0, 1, 0),  # pert 2: DNA CRISPRa, cell type 0 
      (3, 3, 2, 4, 0),  # pert 3: Chemical inhibitor, cell type 0
]


sample_to_pert = [0, 0, 1, 1, 2, 3]

**Control Cells** 

Next we'll show the control cell values.  Recall that for our inference we pair together a perturbed cell with a random control cell from the same batch.  This allows us to take an approximate change in prediction. Beyond just the control cell expression,  to calculate our predicted change in expression, we run the expression predictor on the control cell's latent representation `z_context`. We'll discuss the calculation more but to show this we'll also have the predicted control expression. 

For the predicted, we'll show a minor shift to highlight that often the prediction is not perfect. 

In [5]:
real_control = np.array([
    [2.1, 3.4, 1.2, 4.1, 2.6, 3.1, 1.4, 4.6], 
    [1.9, 3.6, 0.8, 3.9, 2.4, 2.9, 1.6, 4.4], 
    [2.0, 3.3, 1.1, 4.2, 2.3, 3.2, 1.3, 4.3], 
    [2.2, 3.7, 0.9, 3.8, 2.7, 2.8, 1.7, 4.7], 
    [2.0, 3.5, 1.0, 4.0, 2.5, 3.0, 1.5, 4.5], 
    [2.0, 3.5, 1.0, 4.0, 2.5, 3.0, 1.5, 4.5], 
])
pred_control = np.array([
    [2.0, 3.3, 1.3, 4.0, 2.5, 3.2, 1.3, 4.5], 
    [1.8, 3.5, 0.9, 3.8, 2.3, 3.0, 1.5, 4.3], 
    [1.9, 3.2, 1.2, 4.1, 2.2, 3.3, 1.2, 4.2], 
    [2.1, 3.6, 1.0, 3.7, 2.6, 2.9, 1.6, 4.6], 
    [1.9, 3.4, 1.1, 3.9, 2.4, 3.1, 1.4, 4.4], 
    [1.9, 3.4, 1.1, 3.9, 2.4, 3.1, 1.4, 4.4], 
])
real_control.shape, pred_control.shape

((6, 8), (6, 8))

**Case Cell** 

Next we'll show the case cell values. In our raw data we have the real expression of the perturbed cell. We pair this together with the linear expression decoder output based on the mean prediction, mu, from the ACPredictor output. The mean prediction is based on the control cell latent representation `z_context` the perturbations. You'll quickly be able to see that there is a difference between the real values and the predicted values.  We've staged the data so that the first two samples predict closely, the next two samples weakly, the fifth sample predicts in the wrong direction, and the final sample over predicts.  You'll see how these calculations flow through. 

In [6]:
real_case = np.array([
    [2.8, 2.3, 1.6, 6.0, 2.0, 4.7, 1.2, 2.9], 
    [2.8, 2.2, 1.0, 6.0, 2.0, 4.3, 1.6, 2.5], 
    [2.1, 3.2, 1.2, 4.1, 2.4, 3.3, 1.3, 4.4], 
    [2.3, 3.6, 0.9, 3.7, 2.7, 2.8, 1.7, 4.7], 
    [2.5, 2.7, 1.6, 3.0, 2.8, 3.7, 1.1, 5.7], 
    [2.3, 3.1, 1.2, 4.5, 2.2, 3.6, 1.4, 4.9], 
])

pred_case = np.array([
    [2.6, 2.3, 1.8, 5.8, 2.0, 4.6, 1.0, 3.0],  # strong
    [2.6, 2.3, 1.2, 5.8, 2.0, 4.5, 1.4, 2.6],  # strong
    [2.05, 3.05, 1.25, 4.05, 2.3, 3.35, 1.15, 4.3],  # weak
    [2.15, 3.6, 0.95, 3.6, 2.65, 2.95, 1.6, 4.65],  # weak
    [1.6, 3.9, 1.6, 4.5, 2.2, 3.9, 1.7, 3.9],  # inverse
    [2.8, 2.2, 1.7, 5.4, 1.5, 4.9, 1.1, 5.6],  # over
])

real_case.shape, pred_case.shape

((6, 8), (6, 8))

## Calculate Delta

A major component of our expression benchmark is not looking at absolute predictions, but the change in expression. Some claim that this simplifies the task. Biologically, we see this as addressing the import questions: can you predict what will change, in what direction, and by how much. We focus on calculating two sample level differences: 
1. `pred_delta` - the predicted change in expression as caluclated by $\hat{\delta}_g = \hat{x}^{\text{case}}_g - \hat{x}^{\text{ctrl}}_g$. This value compares the predicted perturbed expression (`pred_case`) from the predicted control expression (`pred_control`).  We use the predicted control expression to isolate BioJEPA-AC's learned perturbation effect from any baseline reconstruction error. 
2. `real_delta` - the real change in expression as caluclated by  $\delta_g = x^{\text{case}}_g - x^{\text{ctrl}}_g$. This is our source of truth.

With this calculation you'll see how we end up seeing both increases and decreases in expression by gene. We'll end up comparing these by different slices in our calculations. 

In [7]:
pred_delta = pred_case - pred_control

pred_delta.shape, pred_delta

((6, 8),
 array([[ 0.6 , -1.  ,  0.5 ,  1.8 , -0.5 ,  1.4 , -0.3 , -1.5 ],
        [ 0.8 , -1.2 ,  0.3 ,  2.  , -0.3 ,  1.5 , -0.1 , -1.7 ],
        [ 0.15, -0.15,  0.05, -0.05,  0.1 ,  0.05, -0.05,  0.1 ],
        [ 0.05,  0.  , -0.05, -0.1 ,  0.05,  0.05,  0.  ,  0.05],
        [-0.3 ,  0.5 ,  0.5 ,  0.6 , -0.2 ,  0.8 ,  0.3 , -0.5 ],
        [ 0.9 , -1.2 ,  0.6 ,  1.5 , -0.9 ,  1.8 , -0.3 ,  1.2 ]]))

In [8]:
real_delta = real_case - real_control

real_delta.shape, real_delta

((6, 8),
 array([[ 0.7, -1.1,  0.4,  1.9, -0.6,  1.6, -0.2, -1.7],
        [ 0.9, -1.4,  0.2,  2.1, -0.4,  1.4,  0. , -1.9],
        [ 0.1, -0.1,  0.1, -0.1,  0.1,  0.1,  0. ,  0.1],
        [ 0.1, -0.1,  0. , -0.1,  0. ,  0. ,  0. ,  0. ],
        [ 0.5, -0.8,  0.6, -1. ,  0.3,  0.7, -0.4,  1.2],
        [ 0.3, -0.4,  0.2,  0.5, -0.3,  0.6, -0.1,  0.4]]))

## Calculate Absolutes

You might look at what our model predicted and think "great we have the predicted changes in expression, and we have absolute expression, we're ready to start calculating." This is a bit short sighted though as our absolute predicted expression is based on the models understanding of the predicted control expression.  This fallacy is **the** reason why we focus on predicted changes over just absolute numbers.  If the model can predict the change in experession perfect but just has a different expectation for what the control is, we can have a big issue in our absolute numbers and think we have a crap model when we actually have a great model.  

Because of this we actually create our predicted absolute expression based on the true control and the predicted delta instead of the predicted absolute.  We result in calculating the `pred_abs` as $\hat{x}^{\text{abs}}_g = x^{\text{ctrl}}_g + \hat{\delta}_g$.  

Overall this gives us a way to really evaluate if our model has learned how to shift cells based on perturbation. There is one downside: if we have a very poor prediction, we can end up with a negative gene expression.  While we could handle this as just meaning we're very poor at predicting, we will use a minor bandaid here and just say: the minimum expression we can have is 0.  We do this by applying a clamp in our evals but, since this is just numpy, we'll use a slightly different function. While this will hide very poor negative predictions, we're willing to take this risk with our bandaid. You'll see with our sample data that this isn't an issue since we have all values above zero (on purpose).  We'll also create `real_case` which is just a copy of the case cell expressions.

In [9]:
pred_abs = np.maximum(real_control + pred_delta, 0.0)
pred_abs.shape, pred_abs

((6, 8),
 array([[2.7 , 2.4 , 1.7 , 5.9 , 2.1 , 4.5 , 1.1 , 3.1 ],
        [2.7 , 2.4 , 1.1 , 5.9 , 2.1 , 4.4 , 1.5 , 2.7 ],
        [2.15, 3.15, 1.15, 4.15, 2.4 , 3.25, 1.25, 4.4 ],
        [2.25, 3.7 , 0.85, 3.7 , 2.75, 2.85, 1.7 , 4.75],
        [1.7 , 4.  , 1.5 , 4.6 , 2.3 , 3.8 , 1.8 , 4.  ],
        [2.9 , 2.3 , 1.6 , 5.5 , 1.6 , 4.8 , 1.2 , 5.7 ]]))

In [10]:
real_abs = real_case.copy()
real_abs.shape, real_abs

((6, 8),
 array([[2.8, 2.3, 1.6, 6. , 2. , 4.7, 1.2, 2.9],
        [2.8, 2.2, 1. , 6. , 2. , 4.3, 1.6, 2.5],
        [2.1, 3.2, 1.2, 4.1, 2.4, 3.3, 1.3, 4.4],
        [2.3, 3.6, 0.9, 3.7, 2.7, 2.8, 1.7, 4.7],
        [2.5, 2.7, 1.6, 3. , 2.8, 3.7, 1.1, 5.7],
        [2.3, 3.1, 1.2, 4.5, 2.2, 3.6, 1.4, 4.9]]))

## Sample Level

While we have sample level information, the volume of it can create noisy evals. Because of this we only run a few metrics at the sample level and, the metrics we run, focus on evaluting the expression change metrics (delta). The two we run are mean-squared-error and the pearson correlation coefficient.  MSE is run across all our data while Pearson's $r$ is calculated on only the top 20 differentially expressed genes per sample. 

We'll dive into each calucation

### Mean Squared Error (MSE)

Our first calculation will evaluate simply how far off are our expression change predictions from the real changes.  To do this we calculate the mean squared error as follows:

$$\text{MSE}_{\text{sample}}=\frac{1}{N}\sum_{i=1}^{N}\frac{1}{G}\sum_{g=1}^{G}(\hat{\delta}{i,g} - \delta{i,g})^2$$ 

If you look at the formula, you can see that this calculation will quickly be dominated by the largest expression changes. This is a large reason why we log normalize expression counts, that way large order of magnitude changes do not dominate our optimization.  This is especially important since in many of our perturbation cases a vast majority of our genes will have very minor changes and we need to make sure we're able to predit that just as well as large changes. 

After we calculate per sample MSE, we then take a final mean to get the dataset level mean MSE.  We can see that because of how we seeded our data, the inverse prediction (cell 4) and over prediction (cell 5) have large MSE while the strong and weak predictions have small MSE.  Also note that all MSE are positive so they more indicate the order of maginute of the error, and not the direction.

In [11]:
per_sample_mse = np.mean((pred_delta - real_delta)**2, axis=1)
per_sample_mse.shape, per_sample_mse

((6,),
 array([0.0175   , 0.0175   , 0.001875 , 0.0028125, 1.0675   , 0.58     ]))

In [12]:
sample_mse = np.mean(per_sample_mse)
sample_mse

np.float64(0.28119791666666666)

### Top 20 DEG Pearson's $r$

We next calculate the Pearson's correlation coefficient, $r$, per sample. Pearson's $r$ measures how linearly correlated the predicted and real expression delta profiles are across genes for each sample, regardless of scale. The value of this eval is that if our expression is very linearly correlated, but the  values are off, our model is still useful as it's learned the profile and we just need to find the right scaler to calculate it. For our sample level Pearsons, since this is a benchmark that we compare against other models, we calculate per sample but restrict to the top 20 differentially expressed genes. We pull out the largest (by absolute value) top 20 gene expression changes from our real values real_delta, and then compare those against the calculated changes for those same genes and then take the dataset mean. The resulting calculation is:

$$r_{\text{sample}} = \frac{1}{N}\sum_{i=1}^{N}\frac{\sum_{k=1}^{K}(\hat{\delta}{i,k} - \bar{\hat{\delta}}i)(\delta{i,k} - \bar{\delta}i)}{\sqrt{\sum{k=1}^{K}(\hat{\delta}{i,k} - \bar{\hat{\delta}}i)^{2}} ;\sqrt{\sum{k=1}^{K}(\delta_{i,k} - \bar{\delta}_i)^{2}}}$$

where $K=20$ genes are selected per sample as the largest $|\delta_{i,g}|$. For this example, since we only have 8 genes, instead of taking the top 20, we'll show it by taking only the top 3.

In [13]:
per_sample_corr = []
for i in range(num_cells):
    print(f'----CELL {i}----')
    top_20_idx = np.argsort(np.abs(real_delta[i]))[-TOP_K:]
    pred_top, real_top = pred_delta[i][top_20_idx], real_delta[i][top_20_idx]
    print(f'Top {TOP_K} expression deltas: pred {pred_top} | real {real_top}')
    corr, _ = pearsonr(pred_top, real_top)
    p_corr = 0.0 if np.isnan(corr) else float(corr)
    print(f'Pearson\'s r {p_corr}')
    per_sample_corr.append(p_corr)

per_sample_corr = np.array(per_sample_corr)
per_sample_corr.shape, per_sample_corr

----CELL 0----
Top 3 expression deltas: pred [ 1.4 -1.5  1.8] | real [ 1.6 -1.7  1.9]
Pearson's r 0.9993477847421193
----CELL 1----
Top 3 expression deltas: pred [ 1.5 -1.7  2. ] | real [ 1.4 -1.9  2.1]
Pearson's r 0.9992110001785024
----CELL 2----
Top 3 expression deltas: pred [ 0.1  -0.05  0.1 ] | real [ 0.1 -0.1  0.1]
Pearson's r 1.0
----CELL 3----
Top 3 expression deltas: pred [ 0.05 -0.1   0.  ] | real [ 0.1 -0.1 -0.1]
Pearson's r 0.7559289460184526
----CELL 4----
Top 3 expression deltas: pred [ 0.5  0.6 -0.5] | real [-0.8 -1.   1.2]
Pearson's r -1.0
----CELL 5----
Top 3 expression deltas: pred [1.2 1.5 1.8] | real [0.4 0.5 0.6]
Pearson's r 1.0


((6,),
 array([ 0.99934778,  0.999211  ,  1.        ,  0.75592895, -1.        ,
         1.        ]))

In [14]:
sample_corr = np.mean(per_sample_corr)
sample_corr

np.float64(0.6257479551565125)

## Perturbation Level


Our next set of evals is done at the perturbation level. As a reminder, a we consider a perturbation to be unique if it's the same sequence, target, mode, modality applied to the same cell type.  To analyze per pertubation, we review all of the expression data we have per perturbation and then take the mean to get a single value per perturbation, giving us per gene expression data for each perturbation.  

Once we have per perturbation values, we can calcualte our evaluation. We run a number of per perturbation evaluations including MSE, $R^2$ correlation, and Pearson's $r$.  We run $R^2$ and Pearson's $r$ on both all the genes and the top 50 DEGs. We use "top 50" to align with common industry benchmarks in other papers. 

We'll first aggregate our data. 

### Aggregate Data Per Perturbation



In [ ]:
for pert_idx in range(unique_perts):                                            
    mask = [i for i, p in enumerate(sample_to_pert) if p == pert_idx]           
    mean_pert_pred_delta[pert_idx] = pred_delta[mask].mean(axis=0)                   
    mean_pert_real_delta[pert_idx] = real_delta[mask].mean(axis=0)
    mean_pert_pred_abs[pert_idx] = pred_abs[mask].mean(axis=0)
    mean_pert_real_abs[pert_idx] = real_abs[mask].mean(axis=0)
    mean_pert_real_control[pert_idx] = real_control[mask].mean(axis=0)